# REAL ESTATE PRICE PREDICTION



In [1]:
# # Import important libraries
import pandas as pd
import numpy as np 

# Load dataset 
df = pd.read_csv("bengaluru_house_prices.csv")

# Show first 5 rows 
print("First 5 rows : ", df.head())


First 5 rows :                area_type   availability                  location       size  \
0  Super built-up  Area         19-Dec  Electronic City Phase II      2 BHK   
1            Plot  Area  Ready To Move          Chikka Tirupathi  4 Bedroom   
2        Built-up  Area  Ready To Move               Uttarahalli      3 BHK   
3  Super built-up  Area  Ready To Move        Lingadheeranahalli      3 BHK   
4  Super built-up  Area  Ready To Move                  Kothanur      2 BHK   

   society total_sqft  bath  balcony   price  
0  Coomee        1056   2.0      1.0   39.07  
1  Theanmp       2600   5.0      3.0  120.00  
2      NaN       1440   2.0      3.0   62.00  
3  Soiewre       1521   3.0      1.0   95.00  
4      NaN       1200   2.0      1.0   51.00  


In [2]:
# Check dataset shape (rows, columns)
print("Dataset Shape:")
print(df.shape)

Dataset Shape:
(13320, 9)


In [3]:
# Check column names
print("Columns:")
print(df.columns)


Columns:
Index(['area_type', 'availability', 'location', 'size', 'society',
       'total_sqft', 'bath', 'balcony', 'price'],
      dtype='object')


In [4]:
# Check data types
print("Data Types:")
print(df.dtypes)

Data Types:
area_type        object
availability     object
location         object
size             object
society          object
total_sqft       object
bath            float64
balcony         float64
price           float64
dtype: object


In [5]:
# Check missing values
print("Missing Values:")
print(df.isnull().sum())

Missing Values:
area_type          0
availability       0
location           1
size              16
society         5502
total_sqft         0
bath              73
balcony          609
price              0
dtype: int64


In [6]:
# Basic statistics (only numeric columns)
print("Statistical Summary:")
print(df.describe())


Statistical Summary:
               bath       balcony         price
count  13247.000000  12711.000000  13320.000000
mean       2.692610      1.584376    112.565627
std        1.341458      0.817263    148.971674
min        1.000000      0.000000      8.000000
25%        2.000000      1.000000     50.000000
50%        2.000000      2.000000     72.000000
75%        3.000000      2.000000    120.000000
max       40.000000      3.000000   3600.000000


In [7]:
# Remove rows where size is null
df = df.dropna(subset=['size'])

# Extract first number from size column
df['bhk'] = df['size'].apply(lambda x: int(x.split(' ')[0]))

print("\nAfter extracting BHK:")
print(df[['size', 'bhk']].head())



After extracting BHK:
        size  bhk
0      2 BHK    2
1  4 Bedroom    4
2      3 BHK    3
3      3 BHK    3
4      2 BHK    2


In [8]:

# Function to convert total_sqft to float
def convert_sqft(x):
    try:
        # If range like "1133-1384"
        if '-' in x:
            tokens = x.split('-')
            return (float(tokens[0]) + float(tokens[1])) / 2
        # Normal number
        return float(x)
    except:
        return None  # If unit present or invalid

# Apply function
df['total_sqft'] = df['total_sqft'].apply(convert_sqft)


In [9]:
# Check how many became null
print("Missing after sqft cleaning:")
print(df['total_sqft'].isnull().sum())

Missing after sqft cleaning:
46


In [10]:
# Drop rows where sqft is still null
df = df.dropna(subset=['total_sqft'])

print("\nDataset shape after sqft cleaning:")
print(df.shape)


Dataset shape after sqft cleaning:
(13258, 10)


In [11]:
# price is in lakhs
# total_sqft is in sqft

df['price_per_sqft'] = (df['price'] * 100000) / df['total_sqft']

print("Price per sqft created successfully")

print("\nBasic Stats of price_per_sqft:")
print(df['price_per_sqft'].describe())


Price per sqft created successfully

Basic Stats of price_per_sqft:
count    1.325800e+04
mean     7.912634e+03
std      1.064936e+05
min      2.678298e+02
25%      4.271229e+03
50%      5.438331e+03
75%      7.313266e+03
max      1.200000e+07
Name: price_per_sqft, dtype: float64


In [ ]:


# Remove properties below 1000 Rs per sqft
df = df[df['price_per_sqft'] > 1000]

# Remove properties above 30000 Rs per sqft
df = df[df['price_per_sqft'] < 30000]

print("After logical filtering shape:")
print(df.shape)

print("\nNew price_per_sqft stats:")
print(df['price_per_sqft'].describe())

After logical filtering shape:
(13200, 11)

New price_per_sqft stats:
count    13200.000000
mean      6581.533565
std       3840.181765
min       1166.666667
25%       4268.165119
50%       5425.015738
75%       7277.628032
max      29629.629630
Name: price_per_sqft, dtype: float64


In [ ]:


Q1 = df['price_per_sqft'].quantile(0.25)
Q3 = df['price_per_sqft'].quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

print("Lower Limit:", lower_limit)
print("Upper Limit:", upper_limit)

# Filter dataset
df = df[(df['price_per_sqft'] >= lower_limit) &
        (df['price_per_sqft'] <= upper_limit)]

print("\nAfter IQR filtering shape:")
print(df.shape)

print("\nNew Stats:")
print(df['price_per_sqft'].describe())

Lower Limit: -246.0292512933147
Upper Limit: 11791.82240252801

After IQR filtering shape:
(11969, 11)

New Stats:
count    11969.000000
mean      5574.526880
std       1942.717771
min       1166.666667
25%       4166.666667
50%       5188.679245
75%       6600.000000
max      11785.714286
Name: price_per_sqft, dtype: float64


In [ ]:

# Check sqft per bhk
df = df[df['total_sqft'] / df['bhk'] >= 300]

print("After removing unrealistic BHK houses:")
print(df.shape)

After removing unrealistic BHK houses:
(11648, 11)


In [15]:
print(len(df['location'].unique()))

1126


STEP 7A: Clean Location Names


In [16]:

# remove rows where location is null
df = df.dropna(subset=['location'])

# remove extra spaces
df['location'] = df['location'].apply(lambda x: x.strip())

# count locations
location_stats = df['location'].value_counts()

print(location_stats.head(10))
print("Total unique locations:")
print(len(df['location'].unique()))

location
Whitefield               513
Sarjapur  Road           382
Electronic City          293
Kanakpura Road           269
Thanisandra              234
Yelahanka                204
Uttarahalli              178
Marathahalli             168
Hebbal                   168
Raja Rajeshwari Nagar    167
Name: count, dtype: int64
Total unique locations:
1115


In [17]:
# locations with <=10 houses
location_less_than_10 = location_stats[location_stats <= 10]

print("Locations with <=10 houses:", len(location_less_than_10))

# replace rare locations with "other"
df['location'] = df['location'].apply(
    lambda x: 'other' if x in location_less_than_10 else x
)

print("Unique locations after grouping:")
print(len(df['location'].unique()))

Locations with <=10 houses: 905
Unique locations after grouping:
211


In [ ]:


df = df.drop(['size', 'society', 'price_per_sqft'], axis=1)

print("Columns after dropping:")
print(df.columns)

print("Dataset shape:")
print(df.shape)

Columns after dropping:
Index(['area_type', 'availability', 'location', 'total_sqft', 'bath',
       'balcony', 'price', 'bhk'],
      dtype='object')
Dataset shape:
(11647, 8)


In [ ]:

df = df.drop(['availability'], axis=1)

print("Columns after dropping availability:")
print(df.columns)

print("Dataset shape:")
print(df.shape)

Columns after dropping availability:
Index(['area_type', 'location', 'total_sqft', 'bath', 'balcony', 'price',
       'bhk'],
      dtype='object')
Dataset shape:
(11647, 7)


In [ ]:

# Convert categorical columns into dummy variables
df = pd.get_dummies(df, columns=['area_type', 'location'], drop_first=True)

print("Dataset after encoding:")
print(df.head())

print("New dataset shape:")
print(df.shape)

Dataset after encoding:
   total_sqft  bath  balcony   price  bhk  area_type_Carpet  Area  \
0      1056.0   2.0      1.0   39.07    2                   False   
1      2600.0   5.0      3.0  120.00    4                   False   
2      1440.0   2.0      3.0   62.00    3                   False   
3      1521.0   3.0      1.0   95.00    3                   False   
4      1200.0   2.0      1.0   51.00    2                   False   

   area_type_Plot  Area  area_type_Super built-up  Area  \
0                 False                            True   
1                  True                           False   
2                 False                           False   
3                 False                            True   
4                 False                            True   

   location_2nd Phase Judicial Layout  location_5th Phase JP Nagar  ...  \
0                               False                        False  ...   
1                               False                   

In [ ]:

df['bath'] = df['bath'].fillna(df['bath'].median())
df['balcony'] = df['balcony'].fillna(df['balcony'].median())

# confirm no missing values
print(df.isnull().sum())

total_sqft                     0
bath                           0
balcony                        0
price                          0
bhk                            0
                              ..
location_Yelahanka             0
location_Yelahanka New Town    0
location_Yelenahalli           0
location_Yeshwanthpur          0
location_other                 0
Length: 218, dtype: int64


In [ ]:

from sklearn.model_selection import train_test_split

# X = features
X = df.drop('price', axis=1)

# y = target
y = df['price']

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

# split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Feature shape: (11647, 217)
Target shape: (11647,)
Training data: (9317, 217)
Testing data: (2330, 217)


In [ ]:

from sklearn.linear_model import LinearRegression
y = y.value_counts()

model = LinearRegression()

# train model
model.fit(X_train, y_train)

# check accuracy
score = model.score(X_test, y_test)

print("Linear Regression R² Score:", score)

Linear Regression R² Score: 0.49663933055029696


In [ ]:

from sklearn.tree import DecisionTreeRegressor

dt_model = DecisionTreeRegressor(random_state=42)

dt_model.fit(X_train, y_train)

dt_score = dt_model.score(X_test, y_test)

print("Decision Tree R² Score:", dt_score)

Decision Tree R² Score: 0.5308811534851041


In [ ]:

from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=50,   
    max_depth=None,
    min_samples_split=5,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_score = rf_model.score(X_test, y_test)

print("Random Forest R² Score:", rf_score)

Random Forest R² Score: 0.7642587999649603


In [26]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5]
}

grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
print("Best CV Score:", grid.best_score_)

Best Parameters: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 100}
Best CV Score: 0.7816677378560302


In [27]:

import pickle
import json

# save model
# Save smaller model
with open("real_estate_model.pkl", "wb") as f:
    pickle.dump(rf_model, f)

print("Smaller model saved successfully")

Smaller model saved successfully
